# eXueed — Side-by-Side Pipeline Comparison

This notebook runs the **same query** through all five retrieval / generation pipelines that exist in the repo and prints the results side by side.

| # | Pipeline | Entry point | Used by today |
|---|----------|-------------|---------------|
| 1 | **EnhancedRAGService.query** (the standard landing-page chat path) | `enhanced_rag_service.py:5232` | `POST /api/rag/query/enhanced` — default landing-page `queryEnhanced()` |
| 2 | **ComprehensiveRetriever** (4-phase Qdrant + PG + PTO, CLAUDE.md brief target) | `comprehensive_retrieval.py:102` | Reachable only via `use_study_focused=True` on `/query/enhanced` |
| 3 | **multi_specialty_retrieval** (6-specialty fan-out, no LLM synthesis) | `multi_specialty_retrieval.py:213` | Trial match / patient matching / treatment comparison |
| 4 | **TumorBoardOrchestrator** (multi-specialty fan-out + per-expert LLM assessment) | `tumor_board/orchestrator.py:43` | `POST /api/tumor-board/*` |
| 5 | **QueryIntentService.analyze_query** (Trial-Match / analyzeIntent) | `query_intent_service.py:238` | Front-end Trial-Match toggle → `POST /query/analyze-intent` |

You will provide one query text; the notebook runs all five and prints short summaries, then a side-by-side comparison table (top studies, elapsed ms, chunk count, answer).

> **Prereqs:** Qdrant + Postgres credentials, OpenAI key. The repo's `.env` format is expected.

## 1. Get the repo onto the machine and install dependencies

**The repo is private.** You have two options:

* **Option A (easiest):** Upload the whole folder to Colab first (Files → Upload folder), then set `REPO_DIR` below to that path. The clone step is skipped.
* **Option B:** Paste a GitHub Personal Access Token with read access when prompted. Create one here: <https://github.com/settings/tokens?type=beta>.
  - In Colab you can also store it as a secret named `GITHUB_TOKEN` (key icon in the left sidebar) and the cell will pick it up automatically.

If you're running locally, everything below just no-ops — it only clones when it detects Colab.

In [ ]:
import os, sys, subprocess, getpass, shlex

REPO_OWNER = "alexandrahalfon"
REPO_NAME  = "exueed-updated"
BRANCH     = "claude/tumor-board-architecture-update-jcvRI"  # change if needed

ON_COLAB = "google.colab" in sys.modules
REPO_DIR = f"/content/{REPO_NAME}" if ON_COLAB else os.path.abspath("..")

# ── Option A: repo already present (upload to Colab or run locally) ────────────
# If the repo is already on disk, we skip cloning entirely.
already_present = os.path.isdir(os.path.join(REPO_DIR, "src", "api"))

if ON_COLAB and not already_present:
    # ── Option B: clone with a GitHub PAT (repo is private) ────────────────────
    # Paste a PAT that has read access to the repo. Create one at:
    #   https://github.com/settings/tokens?type=beta
    token = os.environ.get("GITHUB_TOKEN")
    if not token:
        try:
            from google.colab import userdata  # type: ignore
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        token = getpass.getpass("GitHub PAT (leave blank if repo is already uploaded): ").strip()
    if not token:
        raise RuntimeError(
            "Repo not on disk and no GITHUB_TOKEN provided.\n"
            f"Either upload the repo to {REPO_DIR} or paste a PAT above."
        )
    url = f"https://{token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
    subprocess.check_call(["git", "clone", "--branch", BRANCH, "--depth", "1", url, REPO_DIR])

if ON_COLAB:
    subprocess.check_call(["pip", "install", "-q",
        "qdrant-client", "openai", "python-dotenv", "pydantic", "pydantic-settings",
        "fastapi", "asyncpg", "psycopg2-binary", "sentence-transformers",
        "numpy", "scikit-learn", "tiktoken", "rapidfuzz", "httpx",
    ])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("Repo dir:", REPO_DIR)
print("Contents:", sorted(os.listdir(REPO_DIR))[:10])


## 2. Credentials

Fill these in **before** importing any service module — the config singleton reads `os.environ` at import time. On Colab, use `google.colab.userdata` or paste values directly.

In [ ]:
import os

# ── REQUIRED ────────────────────────────────────────────────────────────
os.environ["OPENAI_API_KEY"]  = os.environ.get("OPENAI_API_KEY",  "sk-...")
os.environ["QDRANT_URL"]      = os.environ.get("QDRANT_URL",      "https://<your-qdrant>.cloud.qdrant.io")
os.environ["QDRANT_API_KEY"]  = os.environ.get("QDRANT_API_KEY",  "...")
os.environ["QDRANT_COLLECTION"] = os.environ.get("QDRANT_COLLECTION", "exueed_kb_latest")

# ── OPTIONAL (Postgres — only Pipeline 2 uses it for the PG branch; set empty to skip) ──
os.environ.setdefault("POSTGRES_HOST",     "34.21.60.224")
os.environ.setdefault("POSTGRES_PORT",     "5432")
os.environ.setdefault("POSTGRES_USER",     "postgres")
os.environ.setdefault("POSTGRES_PASSWORD", "")
os.environ.setdefault("POSTGRES_DATABASE", "display-study-details")

assert os.environ["OPENAI_API_KEY"].startswith("sk-"), "Set OPENAI_API_KEY"
assert os.environ["QDRANT_URL"].startswith("http"), "Set QDRANT_URL"
print("Credentials loaded. Collection:", os.environ["QDRANT_COLLECTION"])

## 2b. Ensure Qdrant payload indexes exist (one-time, idempotent)

Creates / backfills payload indexes on the live collection so the typed `MatchAny` filters used by `comprehensive_retrieval.py` hit indexes instead of scanning. Safe to re-run — Qdrant silently accepts re-indexing an already-indexed field.

In [ ]:
from src.ingestion.qdrant_client import QdrantIngestionClient

QdrantIngestionClient().ensure_collection()   # idempotent; safe to re-run

## 3. Test query

Edit the string below. The same text is passed to every pipeline. Use a realistic patient narrative to exercise the LLM-extraction / inference / PTO paths.

In [ ]:
TEST_QUERY = (
    "80 y.o. male non-smoker with PMH HTN, Hep C, BPH, CKD, and recurrent SCC of the "
    "left oral tongue (CPS 100) progressing on pembrolizumab, no longer a surgical "
    "candidate, with radiographic concern for right-ventricle metastasis. "
    "What is the best next-line systemic therapy?"
)
CATEGORY = None   # e.g. "head_neck" — leave None to let each pipeline infer
TOP_K    = 5
print(TEST_QUERY)

## 4. Pipeline runners

Each cell below defines a `run_pipelineX()` coroutine that returns a normalised dict:
```
{ "name": str, "elapsed_ms": float, "answer": str,
  "studies": [{doc_id, title, score, source}],
  "chunks": int, "raw": <original object> }
```

In [ ]:
import asyncio, time, json
from typing import Any, Dict, List

def _summarise_studies(items, limit=5):
    out = []
    for s in items[:limit]:
        out.append({
            "doc_id": (s.get("doc_id") or "")[:40],
            "title":  (s.get("title")  or "")[:80],
            "score":  round(float(s.get("score") or s.get("rerank_score") or 0), 3),
            "source": s.get("source", ""),
        })
    return out

### 4.1 Pipeline 1 — `EnhancedRAGService.query()` (the standard landing-page chat path)

In [ ]:
async def run_pipeline1_enhanced_rag(query: str, category=None, top_k=5) -> Dict[str, Any]:
    from src.api.services.enhanced_rag_service import get_enhanced_rag_service
    svc = get_enhanced_rag_service()
    t0 = time.perf_counter()
    result = await svc.query(
        question=query,
        query_mode="hybrid",
        top_k=top_k,
        category=category,
        use_site_inference=True,
        use_study_focused=False,   # landing-page default
    )
    elapsed = (time.perf_counter() - t0) * 1000
    retrieval = result.get("retrieval_results") or result.get("evidence") or []
    # Dedupe to study level
    seen, studies = set(), []
    for e in retrieval:
        did = e.get("doc_id")
        if did and did not in seen:
            seen.add(did)
            studies.append(e)
    return {
        "name": "1. EnhancedRAGService.query (landing-page path)",
        "elapsed_ms": elapsed,
        "answer": result.get("answer") or result.get("justification") or "",
        "studies": _summarise_studies(studies),
        "chunks":  len(retrieval),
        "raw": result,
    }

### 4.2 Pipeline 2 — `ComprehensiveRetriever.retrieve_comprehensive()` (4-phase)

In [ ]:
async def run_pipeline2_comprehensive(query: str, category=None, top_k=5) -> Dict[str, Any]:
    from src.api.services.comprehensive_retrieval import (
        get_comprehensive_retriever, convert_to_rag_evidence,
    )
    retriever = get_comprehensive_retriever()
    t0 = time.perf_counter()
    result = await retriever.retrieve_comprehensive(
        query_text=query,
        max_studies=top_k,
        chunks_per_study=6,
        category=category,
    )
    elapsed = (time.perf_counter() - t0) * 1000
    evidence, meta = convert_to_rag_evidence(result, max_chunks=top_k * 6)
    studies = [{
        "doc_id": s.doc_id, "title": s.title,
        "score": getattr(s, "rerank_score", None) or getattr(s, "initial_score", 0),
        "source": getattr(s, "source", ""),
    } for s in result.studies]
    return {
        "name": "2. ComprehensiveRetriever (Qdrant + PG + PTO, 4-phase)",
        "elapsed_ms": elapsed,
        "answer": "(retrieval only — no LLM synthesis in this pipeline)",
        "studies": _summarise_studies(studies),
        "chunks":  len(evidence),
        "raw": result,
    }

### 4.3 Pipeline 3 — `multi_specialty_retrieval.retrieve_evidence_multispecialty()`

Six specialty agents (med/rad/surg onc, path-molecular, radiology, palliative) fan out and merge — no LLM synthesis step.

In [ ]:
async def run_pipeline3_multi_specialty(query: str, category=None, top_k=5) -> Dict[str, Any]:
    from src.api.services.multi_specialty_retrieval import retrieve_evidence_multispecialty
    t0 = time.perf_counter()
    ms = await retrieve_evidence_multispecialty(
        case_text=query,
        query_type="treatment_recommendation",
        category=category,
        max_studies=top_k,
    )
    elapsed = (time.perf_counter() - t0) * 1000
    merged = getattr(ms, "merged_studies", []) or []
    studies = [{
        "doc_id": getattr(s, "doc_id", ""),
        "title":  getattr(s, "title", ""),
        "score":  getattr(s, "score", 0),
        "source": ",".join(sorted(getattr(s, "specialties", []) or [])),
    } for s in merged]
    return {
        "name": "3. multi_specialty_retrieval (6-agent fan-out)",
        "elapsed_ms": elapsed,
        "answer": "(retrieval only — stops before LLM expert assessment)",
        "studies": _summarise_studies(studies),
        "chunks":  sum(len(getattr(s, "chunks", []) or []) for s in merged),
        "raw": ms,
    }

### 4.4 Pipeline 4 — `TumorBoardOrchestrator.present_case()` (multi-specialty + LLM assessments)

In [ ]:
async def run_pipeline4_tumor_board(query: str, category=None, top_k=5) -> Dict[str, Any]:
    from src.api.services.tumor_board.orchestrator import get_tumor_board_orchestrator
    orch = get_tumor_board_orchestrator()
    t0 = time.perf_counter()
    report = await orch.present_case(case_text=query, query_type="treatment_recommendation")
    elapsed = (time.perf_counter() - t0) * 1000

    # ExpertAssessment fields (src/api/services/tumor_board/base_agent.py:60-80):
    #   recommendation / recommendation_text / confidence / key_questions /
    #   supporting_studies / conflicting_studies / next_steps / sub_queries /
    #   skipped / skip_reason / error / elapsed_ms
    # StudyCitation fields: doc_id, title, citation, year, relevance_score, snippet
    lines = []
    all_studies = []
    for a in report.expert_assessments:
        tag = "SKIP" if a.skipped else ("ERR" if a.error else a.recommendation)
        text = a.recommendation_text or ("" if not a.error else f"error: {a.error}")
        lines.append(f"[{a.display_name} — {tag} conf={a.confidence:.2f}] {text[:260]}")
        for s in list(a.supporting_studies or []):
            all_studies.append({
                "doc_id": s.doc_id, "title":  s.title,
                "score":  s.relevance_score, "source": a.specialty,
            })
        for s in list(a.conflicting_studies or []):
            all_studies.append({
                "doc_id": s.doc_id, "title":  s.title,
                "score":  s.relevance_score, "source": f"{a.specialty}:conflicting",
            })
    return {
        "name": "4. TumorBoardOrchestrator (6 specialists + LLM assessments)",
        "elapsed_ms": elapsed,
        "answer": "\n".join(lines),
        "studies": _summarise_studies(all_studies, limit=10),
        "chunks":  len(all_studies),
        "raw": report,
    }


### 4.5 Pipeline 5 — `QueryIntentService.analyze_query()` (Trial Match / `/analyze-intent`)

This is what the frontend Trial-Match toggle hits. Forces patient-profile extraction + trial matching.

In [ ]:
async def run_pipeline5_trial_match(query: str, category=None, top_k=5) -> Dict[str, Any]:
    from src.api.services.query_intent_service import get_query_intent_service
    svc = get_query_intent_service()
    t0 = time.perf_counter()
    result = await svc.analyze_query(
        query=query,
        find_matching_trials=True,
        force_trial_match=True,
    )
    elapsed = (time.perf_counter() - t0) * 1000
    trials = result.matching_trials or []
    studies = [{
        "doc_id": getattr(t, "doc_id", "") or getattr(t, "study_id", ""),
        "title":  getattr(t, "title", "") or getattr(t, "study_title", ""),
        "score":  getattr(t, "match_score", 0) or getattr(t, "score", 0),
        "source": getattr(t, "match_reason", "")[:40] if hasattr(t, "match_reason") else "",
    } for t in trials]
    return {
        "name": "5. QueryIntentService (Trial-Match / analyzeIntent)",
        "elapsed_ms": elapsed,
        "answer": result.formatted_response or "(no formatted response)",
        "studies": _summarise_studies(studies, limit=10),
        "chunks":  len(trials),
        "raw": result,
    }

## 5. Run all five in parallel and print a comparison

In [ ]:
async def run_all(query: str, category=None, top_k=5):
    runners = [
        run_pipeline1_enhanced_rag,
        run_pipeline2_comprehensive,
        run_pipeline3_multi_specialty,
        run_pipeline4_tumor_board,
        run_pipeline5_trial_match,
    ]
    results = []
    for fn in runners:
        try:
            results.append(await fn(query, category, top_k))
        except Exception as e:
            import traceback
            results.append({
                "name": fn.__name__,
                "elapsed_ms": 0,
                "answer": f"ERROR: {type(e).__name__}: {e}",
                "studies": [],
                "chunks": 0,
                "traceback": traceback.format_exc(),
            })
    return results

results = await run_all(TEST_QUERY, CATEGORY, TOP_K)

In [ ]:
# Headline table
print(f"{'Pipeline':60} {'ms':>8} {'#chunks':>8} {'#studies':>9}")
print("-" * 90)
for r in results:
    print(f"{r['name'][:58]:60} {r['elapsed_ms']:>8.0f} {r['chunks']:>8} {len(r['studies']):>9}")

In [ ]:
# Per-pipeline detail
for r in results:
    print("\n" + "=" * 90)
    print(r["name"])
    print("=" * 90)
    print(f"Elapsed: {r['elapsed_ms']:.0f} ms   |   chunks: {r['chunks']}   |   studies: {len(r['studies'])}")
    if r.get("traceback"):
        print(r["traceback"]); continue
    print("\nTop studies:")
    for s in r["studies"]:
        print(f"  {s['score']:>6}  [{s['source'][:24]:24}]  {s['title'][:70]}")
    print("\nAnswer:")
    ans = r["answer"] or ""
    print(ans[:2000] + ("…" if len(ans) > 2000 else ""))

In [ ]:
# Study-overlap matrix: which pipelines surfaced the same doc_ids?
from collections import defaultdict
by_pipeline = {r["name"]: {s["doc_id"] for s in r["studies"] if s["doc_id"]} for r in results}
all_ids = set().union(*by_pipeline.values())
print(f"Unique doc_ids across all pipelines: {len(all_ids)}\n")
rows = []
for did in all_ids:
    rows.append([did[:30]] + ["Y" if did in by_pipeline[n] else "." for n in by_pipeline])
header = ["doc_id"] + [n[:12] for n in by_pipeline]
print(" | ".join(f"{h:30}" if i == 0 else f"{h:12}" for i, h in enumerate(header)))
print("-" * (32 + 14 * len(by_pipeline)))
for row in rows:
    print(" | ".join(f"{c:30}" if i == 0 else f"{c:12}" for i, c in enumerate(row)))

## 6. Notes & caveats

- **Pipeline 1** is the only one actually wired into the landing-page chat today. Its `use_study_focused=False` skips the comprehensive path.
- **Pipeline 2** (`ComprehensiveRetriever`) will attempt Qdrant + Postgres + PTO in parallel. If Postgres creds aren't set, it continues with Qdrant + PTO only.
- **Pipeline 3** (multi-specialty) and **Pipeline 4** (tumor board) share the same fan-out — #4 adds a per-expert LLM call, so it is the slowest and most token-hungry.
- **Pipeline 5** (`analyze_query`) re-uses multi-specialty internally for trial matching; it also runs LLM intent detection up front.
- If a pipeline raises at import time (e.g. missing env var), the `except` block in `run_all` preserves the stack trace so you can debug without losing the other pipelines' outputs.
- To compare on different queries, edit `TEST_QUERY` and re-run cells 3 → 5.